In [1]:
import os
import time
import datetime
import pandas as pd
from dotenv import load_dotenv
from datasets import load_dataset
import openai
import anthropic
from google import genai
from google.genai import types
import mlflow
from rouge_score import rouge_scorer

load_dotenv()

if "GOOGLE_API_KEY" in os.environ:
    os.environ.pop("GOOGLE_API_KEY")

openai_client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
anthropic_client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

print("Loading CNN/DailyMail dataset...")
dataset = load_dataset("cnn_dailymail", "3.0.0", split="test")
samples = dataset.select(range(100))

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

SYSTEM_PROMPT = "You are a news summarizer. Summarize the article in 2-3 sentences. Return only the summary, no preamble."

def summarize_openai(article):
    start = time.time()
    response = openai_client.chat.completions.create(
        model="gpt-4o",  # FIX
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": article[:3000]}
        ],
        max_tokens=150,
        temperature=0.0  # FIX
    )
    return response.choices[0].message.content.strip(), time.time() - start

def summarize_anthropic(article):
    start = time.time()
    response = anthropic_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=150,
        temperature=0.0,  # FIX: match GPT-4o/Gemini's greedy decoding for a fair comparison
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": article[:3000]}]
    )
    return response.content[0].text.strip(), time.time() - start

def summarize_gemini(article):
    for attempt in range(4):
        start = time.time()  # FIX: reset per attempt so retry backoff sleep isn't counted as latency
        try:
            response = gemini_client.models.generate_content(
                model="gemini-2.5-flash",
                contents=f"{SYSTEM_PROMPT}\n\n{article[:3000]}",
                config=types.GenerateContentConfig(
                    max_output_tokens=150,
                    temperature=0.0,
                    thinking_config=types.ThinkingConfig(thinking_budget=0)
                )
            )
            return response.text.strip() if response.text else "", time.time() - start
        except Exception as e:
            if "503" in str(e) and attempt < 3:
                time.sleep((attempt + 1) * 3)
                continue
            print(f"Gemini error: {e}")
            return "", time.time() - start

def compute_rouge(hypothesis, reference):
    scores = scorer.score(reference, hypothesis)
    return scores["rouge1"].fmeasure, scores["rouge2"].fmeasure, scores["rougeL"].fmeasure

mlflow.set_experiment("cnn_dailymail_baseline")
results = []

with mlflow.start_run(run_name="summarization_baseline_v2_fixed"):
    mlflow.log_param("sample_count", 100)
    mlflow.log_param("openai_model", "gpt-4o")
    mlflow.log_param("anthropic_model", "claude-haiku-4-5-20251001")
    mlflow.log_param("gemini_model", "gemini-2.5-flash")

    oai_r1s, oai_r2s, oai_rLs, oai_lats = [], [], [], []
    ant_r1s, ant_r2s, ant_rLs, ant_lats = [], [], [], []
    gem_r1s, gem_r2s, gem_rLs, gem_lats = [], [], [], []

    print("Evaluating...")
    for i, sample in enumerate(samples):
        article   = sample["article"]
        reference = sample["highlights"]

        oai_summary, oai_lat = summarize_openai(article)
        ant_summary, ant_lat = summarize_anthropic(article)
        gem_summary, gem_lat = summarize_gemini(article)

        oai_r1, oai_r2, oai_rL = compute_rouge(oai_summary, reference)
        ant_r1, ant_r2, ant_rL = compute_rouge(ant_summary, reference)
        gem_r1, gem_r2, gem_rL = compute_rouge(gem_summary, reference)

        oai_r1s.append(oai_r1); oai_r2s.append(oai_r2); oai_rLs.append(oai_rL); oai_lats.append(oai_lat)
        ant_r1s.append(ant_r1); ant_r2s.append(ant_r2); ant_rLs.append(ant_rL); ant_lats.append(ant_lat)
        gem_r1s.append(gem_r1); gem_r2s.append(gem_r2); gem_rLs.append(gem_rL); gem_lats.append(gem_lat)

        results.append({
            "article_snippet":    article[:80],
            "reference_summary":  reference,
            "openai_summary":     oai_summary,
            "anthropic_summary":  ant_summary,
            "gemini_summary":     gem_summary,
            "openai_latency_s":   round(oai_lat, 3),
            "anthropic_latency_s":round(ant_lat, 3),
            "gemini_latency_s":   round(gem_lat, 3),
            "openai_rougeL":      round(oai_rL, 4),
            "anthropic_rougeL":   round(ant_rL, 4),
            "gemini_rougeL":      round(gem_rL, 4),
        })

        if i % 10 == 0:
            print(f"Progress: {i}/100")

    avg = lambda lst: sum(lst) / len(lst)

    mlflow.log_metric("openai_avg_rouge1",      avg(oai_r1s))
    mlflow.log_metric("openai_avg_rouge2",      avg(oai_r2s))
    mlflow.log_metric("openai_avg_rougeL",      avg(oai_rLs))
    mlflow.log_metric("anthropic_avg_rouge1",   avg(ant_r1s))
    mlflow.log_metric("anthropic_avg_rouge2",   avg(ant_r2s))
    mlflow.log_metric("anthropic_avg_rougeL",   avg(ant_rLs))
    mlflow.log_metric("gemini_avg_rouge1",      avg(gem_r1s))
    mlflow.log_metric("gemini_avg_rouge2",      avg(gem_r2s))
    mlflow.log_metric("gemini_avg_rougeL",      avg(gem_rLs))
    mlflow.log_metric("openai_avg_latency_s",   avg(oai_lats))
    mlflow.log_metric("anthropic_avg_latency_s",avg(ant_lats))
    mlflow.log_metric("gemini_avg_latency_s",   avg(gem_lats))

    print(f"\n{'='*55}")
    print(f"{'Metric':<22} {'GPT-4o':>8} {'Haiku':>10} {'Gemini':>10}")
    print(f"{'='*55}")
    print(f"{'Avg Latency (s)':<22} {avg(oai_lats):>8.3f} {avg(ant_lats):>10.3f} {avg(gem_lats):>10.3f}")
    print(f"{'ROUGE-1':<22} {avg(oai_r1s):>8.3f} {avg(ant_r1s):>10.3f} {avg(gem_r1s):>10.3f}")
    print(f"{'ROUGE-2':<22} {avg(oai_r2s):>8.3f} {avg(ant_r2s):>10.3f} {avg(gem_r2s):>10.3f}")
    print(f"{'ROUGE-L':<22} {avg(oai_rLs):>8.3f} {avg(ant_rLs):>10.3f} {avg(gem_rLs):>10.3f}")
    print(f"{'='*55}")

os.makedirs("../logs", exist_ok=True)
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = f"../logs/cnn_dailymail_baseline_{timestamp}.csv"
pd.DataFrame(results).to_csv(csv_path, index=False)
print(f"Saved to {csv_path}")

Loading CNN/DailyMail dataset...
Evaluating...
Progress: 0/100
Progress: 10/100
Progress: 20/100
Progress: 30/100
Progress: 40/100
Progress: 50/100
Progress: 60/100
Progress: 70/100
Progress: 80/100
Progress: 90/100

Metric                   GPT-4o      Haiku     Gemini
Avg Latency (s)           2.373      2.042      1.107
ROUGE-1                   0.323      0.309      0.324
ROUGE-2                   0.115      0.107      0.117
ROUGE-L                   0.221      0.209      0.222
Saved to ../logs/cnn_dailymail_baseline_20260526_134255.csv
